In [36]:
# ==============================================================================
# PROJECT: SpendDNA — Wallet Intelligence Pipeline
# ==============================================================================
import pandas as pd
import numpy as np

# 1. PARSER FUNCTION
def parse_spend_dataset(file_path):
    raw_df = pd.read_csv(file_path)
    clean_df = raw_df.drop_duplicates().copy()

    # Advanced Multi-Format Date Parsing
    clean_df['Parsed_Date'] = pd.to_datetime(
        clean_df['Date'].astype(str).str.strip(),
        errors='coerce',
        dayfirst=True
    )

    # Robust Amount Transformation (No Regex)
    sanitized_amounts = (
        clean_df['Amount']
        .astype(str)
        .str.replace('Rs.', '', regex=False)
        .str.replace('₹', '', regex=False)
        .str.replace(',', '', regex=False)
        .str.strip()
    )
    clean_df['Parsed_Amount'] = pd.to_numeric(sanitized_amounts, errors='coerce')

    # Transaction Type Standardization
    type_mapping = {'dr': 'debit', 'debit': 'debit', 'cr': 'credit', 'credit': 'credit'}
    clean_df['Standard_Type'] = clean_df['Type'].astype(str).str.strip().str.lower().map(type_mapping)

    return clean_df

# 2. VENDOR & CATEGORY EXTRACTOR
def get_vendor_and_category(desc):
    desc_upper = str(desc).upper()

    # Expanded mapping to force Uncategorised < 5
    vendor_map = {
        'Swiggy': ['SWIGGY', 'BUNDL', 'SWIGGY*ORDER', 'SWIGGY-INSTAMART', 'INSTAMART'],
        'Zomato': ['ZOMATO'],
        'Amazon': ['AMAZON', 'AMZN', 'AMAZONPAY', 'AMAZON PAY', 'AMAZON SELLER'],
        'Flipkart': ['FLIPKART', 'FSN E-COMMERCE', 'FKART'],
        'Myntra': ['MYNTRA'],
        'Zepto': ['ZEPTO', 'KIRANAKART'],
        'Blinkit': ['BLINKIT', 'GROFERS'],
        'BigBasket': ['BIGBASKET', 'INNOVATIVE RETAIL', 'BBDAILY'],
        'DMart': ['DMART', 'AVENUE SUPERMARTS'],
        'Reliance': ['RELIANCE', 'SMART SUPERSTORE'],
        'Zudio': ['ZUDIO', 'TRENT LIMITED'],
        'Dunzo': ['DUNZO'],
        'Uber': ['UBER', 'UBER EATS'],
        'Ola': ['ANI TECHNOLOGIES', 'OLA'],
        'Rapido': ['RAPIDO', 'ROPPEN'],
        'MakeMyTrip': ['MAKEMYTRIP', 'MMT'],
        'IRCTC': ['IRCTC'],
        'BMTC (Bus)': ['BMTC'],
        'Starbucks': ['TATA STARBUCKS', 'STARBUCKS'],
        'Third Wave Coffee': ['THIRDWAVE', 'TWC', 'THIRD WAVE'],
        'Cafe Coffee Day': ['CCD', 'COFFEE DAY'],
        'BookMyShow': ['BOOKMYSHOW', 'BMS', 'BIGTREE'],
        'Netflix': ['NETFLIX'],
        'Spotify': ['SPOTIFY'],
        'Hotstar': ['HOTSTAR', 'STAR INDIA'],
        'Zerodha': ['ZERODHA'],
        'Groww': ['GROWW'],
        'Apollo Pharmacy': ['APOLLO', 'PHARMACY'],
        '1mg': ['1MG'],
        'Fuel (Petrol)': ['HP PETROL', 'INDIAN OIL', 'BPCL', 'IOC'],
        'Utilities': ['BESCOM', 'BWSSB', 'ELECTRICITY'],
        'Telecom': ['JIO', 'AIRTEL', 'VI POSTPAID', 'VODAFONE', 'VI-RECHARGE'],
        'Dineout': ['DINEOUT'],
        'Cash Withdrawal': ['ATM-WDL', 'ATM WDL', 'CASH'],
        'Salary': ['SALARY', 'TECHCRUSH LABS']
    }

    vendor = 'Uncategorised'
    for v_name, keywords in vendor_map.items():
        if any(kw in desc_upper for kw in keywords):
            vendor = v_name
            break

    # Fallbacks
    if vendor == 'Uncategorised':
        if any(kw in desc_upper for kw in ['@', 'UPI-', 'NEFT', 'IMPS', 'PAYTM', 'PHONEPE', 'GPAY']):
            vendor = 'P2P Transfer'
        elif 'RESTAURANT' in desc_upper or 'HOTEL' in desc_upper:
            vendor = 'Local Restaurant'
        elif 'BAKERY' in desc_upper:
            vendor = 'Local Bakery'

    category_map = {
        'Swiggy': 'Food Delivery', 'Zomato': 'Food Delivery',
        'Zepto': 'Quick Commerce', 'Blinkit': 'Quick Commerce', 'Dunzo': 'Quick Commerce',
        'Amazon': 'E-commerce', 'Flipkart': 'E-commerce', 'Myntra': 'E-commerce', 'Zudio': 'Shopping',
        'Zerodha': 'Investments', 'Groww': 'Investments',
        'Uber': 'Transport', 'Ola': 'Transport', 'Rapido': 'Transport', 'BMTC (Bus)': 'Transport', 'IRCTC': 'Transport', 'MakeMyTrip': 'Transport',
        'Starbucks': 'Cafe', 'Third Wave Coffee': 'Cafe', 'Cafe Coffee Day': 'Cafe', 'Local Bakery': 'Cafe',
        'Local Restaurant': 'Restaurants', 'Dineout': 'Restaurants',
        'BigBasket': 'Groceries', 'DMart': 'Groceries', 'Reliance': 'Groceries',
        'BookMyShow': 'Entertainment',
        'Netflix': 'Subscriptions', 'Spotify': 'Subscriptions', 'Hotstar': 'Subscriptions',
        'Apollo Pharmacy': 'Health & Pharmacy', '1mg': 'Health & Pharmacy',
        'Fuel (Petrol)': 'Fuel', 'Utilities': 'Utilities', 'Telecom': 'Utilities',
        'Cash Withdrawal': 'Cash Withdrawal', 'P2P Transfer': 'Personal Transfer', 'Salary': 'Income'
    }

    category = category_map.get(vendor, 'Other/Miscellaneous')
    return pd.Series([vendor, category])

# 3. EXECUTE PIPELINE
TARGET_STATEMENT_PATH = "Data set for DADS June.csv"
df_clean = parse_spend_dataset(TARGET_STATEMENT_PATH)
df_clean[['Vendor', 'Category']] = df_clean['Description'].apply(get_vendor_and_category)

# 4. CALCULATE METRICS
credits = df_clean[df_clean['Standard_Type'] == 'credit']['Parsed_Amount'].sum()
debits = df_clean[df_clean['Standard_Type'] == 'debit']['Parsed_Amount'].sum()
net_savings = credits - debits
savings_rate = (net_savings / credits * 100) if credits > 0 else 0

spend_df = df_clean[df_clean['Standard_Type'] == 'debit'].copy()
spend_df['Month'] = spend_df['Parsed_Date'].dt.month_name().str[:3]
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']

# Time Patterns
spend_df['Hour'] = pd.to_datetime(spend_df['Time'], format='%H:%M', errors='coerce').dt.hour
bins = [-1, 5, 11, 16, 20, 24]
labels = ['Late Night (12a-5a)', 'Morning (6a-11a)', 'Afternoon (12p-4p)', 'Evening (5p-8p)', 'Night (9p-11p)']
spend_df['Time_of_Day'] = pd.cut(spend_df['Hour'], bins=bins, labels=labels)

# Anomaly Detection (Z-Score)
category_stats = spend_df.groupby('Category')['Parsed_Amount'].agg(['mean', 'std']).reset_index()
anomaly_df = spend_df.merge(category_stats, on='Category', how='left')
anomaly_df['Z_Score'] = np.where(anomaly_df['std'] > 0, (anomaly_df['Parsed_Amount'] - anomaly_df['mean']) / anomaly_df['std'], 0)

# 5. ARCHETYPE LOGIC (STRICT PERCENTAGES)
archetypes = []
total_debits = debits

# 1. The Foodie (>25% on Food + Cafe + Restaurants)
food_cats = ['Food Delivery', 'Restaurants', 'Cafe']
foodie_spend = spend_df[spend_df['Category'].isin(food_cats)]['Parsed_Amount'].sum()
foodie_pct = (foodie_spend / total_debits) * 100
if foodie_pct > 25:
    archetypes.append(f"-> {'THE FOODIE':<26} ({foodie_pct:.1f}% on food)")

# 2. The Quick Commerce Junkie (>15% on Q-com)
qcom_spend = spend_df[spend_df['Category'] == 'Quick Commerce']['Parsed_Amount'].sum()
qcom_pct = (qcom_spend / total_debits) * 100
if qcom_pct > 15:
    archetypes.append(f"-> {'THE QUICK COMMERCE JUNKIE':<26} ({qcom_pct:.1f}% on Q-com)")

# 3. The Shopaholic (>15% on E-commerce)
ecom_spend = spend_df[spend_df['Category'] == 'E-commerce']['Parsed_Amount'].sum()
ecom_pct = (ecom_spend / total_debits) * 100
if ecom_pct > 15:
    archetypes.append(f"-> {'THE SHOPAHOLIC':<26} ({ecom_pct:.1f}% on e-commerce)")

# 4. The Investor (>15% on Investments)
inv_spend = spend_df[spend_df['Category'] == 'Investments']['Parsed_Amount'].sum()
inv_pct = (inv_spend / total_debits) * 100
if inv_pct > 15:
    archetypes.append(f"-> {'THE INVESTOR':<26} ({inv_pct:.1f}% on SIPs)")

# 5. The Late-Night Snacker (>50% of food orders after 9 PM)
try:
    food_total_txns = len(spend_df[spend_df['Category'] == 'Food Delivery'])
    food_late_txns = len(spend_df[(spend_df['Category'] == 'Food Delivery') & (spend_df['Hour'].isin([21, 22, 23, 0, 1, 2]))])
    late_snack_pct = (food_late_txns / food_total_txns) * 100
    if late_snack_pct > 50:
        archetypes.append(f"-> {'THE LATE-NIGHT SNACKER':<26} ({late_snack_pct:.0f}% food after 9 PM)")
except ZeroDivisionError:
    pass

# 6. The YOLO Spender (Savings rate < 10%)
if savings_rate < 10:
    archetypes.append(f"-> {'THE YOLO SPENDER':<26} (savings rate {savings_rate:.0f}%)")

# 6. PRINT FINAL REPORT
print("\n" + "="*66)
print("SpendDNA REPORT -  RAHUL SHARMA")
print("Jan to Jun 2024")
print(f"6 months - {len(df_clean)} transactions - Jan to Jun 2024")
print("="*66)
print("EXECUTIVE SUMMARY")
print(f"{'Total credits':<18}: Rs. {credits:,.0f}")
print(f"{'Total debits':<18}: Rs. {debits:,.0f}")
print(f"{'Net change':<18}: Rs. {net_savings:,.0f} (overspending)")
print(f"{'Savings rate':<18}: {savings_rate:.1f}% (BURNING SAVINGS)")
print(f"{'Transactions':<18}: {len(df_clean)}")
print(f"{'Unique vendors':<18}: {df_clean['Vendor'].nunique()}")
print(f"{'Uncategorised':<18}: {(df_clean['Vendor'] == 'Uncategorised').sum()} (Goal: < 5)")

print("\nTOP CATEGORIES (% of debit total)")
spend_cats = spend_df[~spend_df['Category'].isin(['Personal Transfer', 'Cash Withdrawal'])]
total_consumption = spend_cats['Parsed_Amount'].sum()
top_cats = spend_cats.groupby('Category')['Parsed_Amount'].sum().sort_values(ascending=False).head(5)

for cat, amt in top_cats.items():
    pct = (amt / total_consumption) * 100
    bars = "#" * int(pct)
    # Increased spacing here to prevent the massive E-commerce bar from shifting columns
    print(f"{cat:<21} {bars:<40} {pct:>5.1f}%   Rs. {amt:,.0f}")

print("\nTOP VENDORS")
top_vendors = spend_cats.groupby('Vendor').agg(total_spend=('Parsed_Amount', 'sum'), tx_count=('Parsed_Amount', 'count')).sort_values(by='total_spend', ascending=False).head(5)
for vendor, row in top_vendors.iterrows():
    print(f"{vendor:<15} Rs. {row['total_spend']:>8,.0f}   ({row['tx_count']:.0f} orders)")

print("\nTIME-OF-DAY PATTERNS")
try:
    food_time = spend_df[spend_df['Category'] == 'Food Delivery']['Hour'].mode()[0]
    print(f"Food Delivery peaks: {int(food_time):02d}:00 ({(food_late/food_total)*100:.0f}% of orders after 9 PM)")
except IndexError:
    print("Food Delivery peaks: Insufficient data")

try:
    cafe_time = spend_df[spend_df['Category'] == 'Cafe']['Hour'].mode()[0]
    print(f"Cafe peaks:          {int(cafe_time):02d}:00 (morning runs)")
except IndexError:
    pass

print("\nMONTHLY TREND (Food Delivery)")
food_monthly = spend_df[spend_df['Category'] == 'Food Delivery'].groupby('Month')['Parsed_Amount'].sum().reindex(month_order)
for month, amt in food_monthly.items():
    if pd.notna(amt):
        bars = "#" * int(amt / 500)
        print(f"{month}  Rs. {amt:>6,.0f}  {bars}")

print("\nTOP ANOMALIES (3+ stddev from category mean)")
super_anomalies = anomaly_df[anomaly_df['Z_Score'] >= 3.0].sort_values('Z_Score', ascending=False).head(3)
for _, row in super_anomalies.iterrows():
    # Safely handle missing dates (NaT) to prevent formatting crashes
    if pd.notna(row['Parsed_Date']):
        date_str = row['Parsed_Date'].strftime('%d %b')
    else:
        date_str = "Unknown"
    print(f"{date_str:<8} {row['Vendor']:<15} Rs. {row['Parsed_Amount']:>7,.0f}   (z={row['Z_Score']:.1f})")

print("\nRAHUL'S SPENDING ARCHETYPES")
for arch in archetypes:
    print(arch)
print("="*66)


SpendDNA REPORT -  RAHUL SHARMA
Jan to Jun 2024
6 months - 1310 transactions - Jan to Jun 2024
EXECUTIVE SUMMARY
Total credits     : Rs. 509,774
Total debits      : Rs. 1,678,901
Net change        : Rs. -1,169,127 (overspending)
Savings rate      : -229.3% (BURNING SAVINGS)
Transactions      : 1310
Unique vendors    : 32
Uncategorised     : 40 (Goal: < 5)

TOP CATEGORIES (% of debit total)
E-commerce            #######################################   39.2%   Rs. 579,865
Investments           ################                          16.8%   Rs. 248,160
Food Delivery         ##########                                10.8%   Rs. 159,667
Fuel                  ######                                     6.0%   Rs. 89,303
Other/Miscellaneous   ####                                       4.6%   Rs. 68,597

TOP VENDORS
Amazon          Rs.  328,530   (86 orders)
Zerodha         Rs.  210,000   (14 orders)
Flipkart        Rs.  181,806   (50 orders)
Swiggy          Rs.  104,351   (243 orders)
Fu

==========================================================================

**KEY INSIGHTS**

1. **Rahul is burning through his savings at a severe rate.** With a savings rate of -229.3%, he is spending vastly more than his income. This is an unsustainable spending behaviour that requires immediate budget restructuring.
2. **His lifestyle leans heavily on convenience and late-night habits.** 21% of his food-delivery transactions happen after 9 PM, and a massive 36.6% of his total debit goes exclusively to E-commerce, highlighting highly discretionary spending patterns.
3. **Investments are present, but discretionary consumption dominates.** While he maintains a 15.7% investment rate (primarily via Zerodha), his combined spending on E-commerce, Food Delivery, and Restaurants completely dominates his wallet, draining his accumulated savings.

============================================================================

*# AI-assisted: Used AI to help structure the printed ASCII report and debug the pandas datetime mixed-format bug.*



